# Solution F2 · Linear PCA against KernelPCA on nonlinear data (and what it costs)

**Type**: Compare

## Objective

Show that on nonlinear data (two concentric circles, two moons) linear PCA does not help a linear classifier, while `KernelPCA` with an RBF kernel makes the problem separable with a few components; choose components and `gamma` on a grid and measure how the computation time grows with the number of examples.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

import time
from sklearn.datasets import make_circles, make_moons
from sklearn.decomposition import PCA, KernelPCA
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import make_pipeline

datasets = {"circles": make_circles(1000, noise=0.05, factor=0.4, random_state=0), "moons": make_moons(1000, noise=0.15, random_state=0)}
cv = StratifiedKFold(5, shuffle=True, random_state=0)

## Commented solution

In [ ]:
base, pca2 = {}, {}
for name, (X, y) in datasets.items():
    base[name] = cross_val_score(LogisticRegression(max_iter=1000), X, y, cv=cv).mean()
    pca2[name] = cross_val_score(make_pipeline(PCA(2), LogisticRegression(max_iter=1000)), X, y, cv=cv).mean()
pd.DataFrame({"logistic": base, "PCA(2)": pca2}).round(3)

In [ ]:
ks, gammas = [2, 5, 10], [1, 3, 10]
grid, best, min_k = {}, {}, {}
for name, (X, y) in datasets.items():
    g = pd.DataFrame(index=ks, columns=gammas, dtype=float)
    for k in ks:
        for gamma in gammas:
            g.loc[k, gamma] = cross_val_score(make_pipeline(KernelPCA(k, kernel="rbf", gamma=gamma), LogisticRegression(max_iter=1000)), X, y, cv=cv).mean()
    grid[name] = g
    best[name] = min((k, gamma) for k in ks for gamma in gammas if g.loc[k, gamma] >= g.to_numpy().max() - 1e-12)
    min_k[name] = min(k for k in ks if g.loc[k].max() > 0.95)
    print(name); print(g.round(3))
print("best:", best, "| minimum k to go over 0.95:", min_k)

In [ ]:
def kpca_time(n):
    X, _ = make_moons(n, noise=0.15, random_state=0)
    t0 = time.time()
    KernelPCA(5, kernel="rbf", gamma=3).fit(X)
    return time.time() - t0


times = pd.Series({n: kpca_time(n) for n in [500, 1000, 2000, 4000]})
exponent = np.polyfit(np.log(times.index.to_numpy(dtype=float)), np.log(times.to_numpy()), 1)[0]
print(times.round(2).to_string())
print(f"the time grows like n^{exponent:.2f}")

## Check

In [ ]:
with verify():
    check(set(base) == {"circles", "moons"} and base["circles"] < 0.55 and pca2["circles"] < 0.55 and 0.8 < base["moons"] < 0.9, "circles: logistic and PCA(2) at chance (below 0.55); moons: the logistic reaches 0.85", "PCA(2) is a rotation: it changes nothing")
    check(all(grid[n].shape == (3, 3) for n in ("circles", "moons")) and grid["circles"].loc[5].max() > 0.99 and grid["moons"].loc[5].max() > 0.96, "KernelPCA with 5 components: circles over 0.99, moons over 0.96", "RBF kernel, k=5")
    check(best["circles"][0] >= 5 and min_k["circles"] == 5 and min_k["moons"] == 5, "5 components are needed (2 is not enough with any gamma)", "the smallest k with accuracy above 0.95")
    check(grid["circles"].loc[2].max() < 0.8 and grid["moons"].loc[2].max() < 0.95, "with 2 components KernelPCA does not solve the problem (circles below 0.8, moons below 0.95)", "two components are not enough")
    check(list(times.index) == [500, 1000, 2000, 4000] and times.is_monotonic_increasing and exponent > 1.3 and times[4000] > 5 * times[1000], "the time grows more than linearly with n (exponent over 1.3; 4000 points over 5 times slower than 1000)", "the kernel matrix is n x n")
done()

## What you learned

- **Linear PCA does not solve nonlinear problems:** on the circles the logistic and `PCA(2)` are at chance (0.46) because two concentric circles are not separable by any line, and rotating the axes does not change the geometry. On the moons 0.865.
- **`KernelPCA` with an RBF kernel does PCA in an implicit space of much higher dimension:** with 5 components the logistic reaches 1.00 on the circles and 0.98 on the moons. With 2 components it is not enough (0.46-0.70 on the circles): the dimension matters. `gamma` (the width of the kernel) is a hyperparameter like
  `k`: but here its influence is small once `k` is sufficient.
- **The cost:** the kernel matrix is `n × n` and the solvers decompose it, so the time grows roughly like the square of `n` (here the measured exponent is 1.9). With 4000 points it is already over ten times slower than with 1000 (the seconds depend on your machine), and with 100 000 examples it is unusable without approximations (Nystroem, random Fourier features).
- The alternatives: **Nystroem** (kernel approximation with a subset), **Isomap** and **t-SNE/UMAP** (for visualization), or a nonlinear model directly (trees, kernel SVM, networks). KernelPCA is a niche choice: interesting for understanding, rarely the most practical.

## Common pitfall

Expecting KernelPCA to be "PCA but better". It has more hyperparameters (kernel, `gamma`), it is not simply invertible (no exact `inverse_transform`), it costs `O(n^2)` in memory and it gives no interpretable loadings.

## Interview question

> *Click on a question to show its answer.*

<details><summary>When is linear PCA not enough?</summary>

When the structure of the data is nonlinear (curved manifolds, nested clusters): two concentric circles, images with deformations. The options are KernelPCA (RBF kernel), Isomap or embedding methods (t-SNE, UMAP) for visualization, autoencoders for compression. Kernel methods
cost O(n^2) in memory and time, so they are used with small datasets or with approximations (Nystroem). And before complicating things, I check whether a nonlinear model directly on the original data already solves the problem.

</details>